In [8]:
import numpy as np

class SimpleEnvironment:
    """A simple 1D grid environment with 5 states (0 to 4).
    State 4 is the terminal state.
    The only action is to move right.
    """
    def __init__(self):
        self.num_states = 5
        self.reset()

    def reset(self):
        self.state = 0
        return self.state

    def step(self, action):
        # Move right regardless of the action for simplicity
        self.state += 1

        # Define rewards
        if self.state == 4:
            reward = 10.0  # Big reward at the goal state
            done = True
        else:
            reward = -1.0  # Step penalty
            done = False

        return self.state, reward, done

def generate_episode(env, policy):
    """Generates an episode using the given policy."""
    episode = []
    state = env.reset()
    done = False

    while not done:
        action = policy(state)
        next_state, reward, done = env.step(action)
        episode.append((state, action, reward))
        state = next_state

    return episode

def first_visit_mc_prediction(env, policy, num_episodes, gamma=1.0):
    """Calculates the state-value function V(s) using First-Visit Monte Carlo Prediction."""
    # Initialize total returns sum and visit counts for each state
    returns_sum = np.zeros(env.num_states)
    returns_count = np.zeros(env.num_states)

    for _ in range(num_episodes):
        # 1. Generate an episode
        episode = generate_episode(env, policy)

        # Keep track of states visited in this specific episode
        visited_states = set()

        # 2. Loop through the steps of the episode
        G = 0
        # Step backwards through the episode to calculate returns efficiently
        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]

            # Update the return G_t = R_{t+1} + gamma * G_{t+1}
            G = gamma * G + reward

            # Check if this is the FIRST visit to the state in this episode
            if state not in visited_states:
                visited_states.add(state)
                returns_sum[state] += G
                returns_count[state] += 1

    # 3. Calculate V(s) as the average return for each state
    # Avoid division by zero for unvisited states
    V = np.zeros(env.num_states)
    for s in range(env.num_states):
        if returns_count[s] > 0:
            V[s] = returns_sum[s] / returns_count[s]

    return V

# --- Execution ---
if __name__ == "__main__":
    # Define a dummy policy (always chooses action 0)
    def dummy_policy(state):
        return 0

    # Initialize environment
    env = SimpleEnvironment()

    # Run First-Visit MC Prediction for 1000 episodes with a discount factor of 0.9
    num_episodes = 1000
    gamma = 0.9

    V_estimates = first_visit_mc_prediction(env, dummy_policy, num_episodes, gamma)

    # Print the results
    print("\nFirst-Visit Monte Carlo State Values:")
    for state, val in enumerate(V_estimates):
        print(f"State {state}: {val:.4f}")


First-Visit Monte Carlo State Values:
State 0: 4.5800
State 1: 6.2000
State 2: 8.0000
State 3: 10.0000
State 4: 0.0000
